# Organize data storage.

## Import Libraries.

In [3]:
import os

import anndata as ad
import numpy as np
import pandas as pd

### Constants for loops to run.

In [4]:
RUN_LOOP2 = False
RUN_LOOP2p5 = False
RUN_LOOP3 = False
RUN_LOOP4 = False
RUN_LOOP5 = True

## Loop 1.

### Read adata for loop 1.

In [3]:
# ---- Define output paths ----
loop1_dump_dir = "../../project_folder/output/loops/data/loop1"
loop1_500k_res_out_path = os.path.join(loop1_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop1_full_res_out_path = os.path.join(loop1_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop1_500k_out_path = os.path.join(loop1_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop1_full_out_path = os.path.join(loop1_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Read Data ----
adata_loop1_500k = ad.read_h5ad(loop1_500k_res_out_path)
adata_loop1_full = ad.read_h5ad(loop1_full_res_out_path)
adata_loop1_concat_500k = ad.read_h5ad(loop1_500k_out_path)
adata_loop1_concat_full = ad.read_h5ad(loop1_full_out_path)

### Get Experiment Numbers for Loop 1.

In [4]:
# ---- Get unique experiment numbers for 500k subset ----
exp_numbers_loop1_500k = adata_loop1_500k.obs["experiment_number"].unique()
exp_numbers_loop1_full = adata_loop1_full.obs["experiment_number"].unique()
exp_numbers_loop1_concat_500k = adata_loop1_concat_500k.obs["experiment_number"].unique()
exp_numbers_loop1_concat_full = adata_loop1_concat_full.obs["experiment_number"].unique()
print(f"Loop 1 500k subset {len(exp_numbers_loop1_500k)} experiments -> {exp_numbers_loop1_500k}")
print(f"Loop 1 full data {len(exp_numbers_loop1_full)} experiments -> {exp_numbers_loop1_500k}")
print(f"Loop 1 Concat 500k subset {len(exp_numbers_loop1_concat_500k)} experiments")
print(f"Loop 1 Concat full data {len(exp_numbers_loop1_concat_full)} experiments")

Loop 1 500k subset 5 experiments -> [208 206 210 207 209]
Loop 1 full data 5 experiments -> [208 206 210 207 209]
Loop 1 Concat 500k subset 122 experiments
Loop 1 Concat full data 122 experiments


## Loop 2.

### Load input data.

In [17]:
if RUN_LOOP2:
    # ---- Load 500k subset ----
    adata_loop2_500k_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop2_logicle_500k.h5ad"
    adata_loop2_500k = ad.read_h5ad(adata_loop2_500k_path)
    print(f"{adata_loop2_500k=}")

    # ---- Load 500k subset ----
    adata_loop2_full_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop2_logicle.h5ad"
    adata_loop2_full = ad.read_h5ad(adata_loop2_full_path)
    print(f"{adata_loop2_full=}")


### Get loop 2 experiment numbers.

In [18]:
if RUN_LOOP2:
    # ---- Get unique experiment numbers from 500k subset ----
    loop2_exp_numbers_500k = adata_loop2_500k.obs.experiment_number.unique().tolist()
    loop2_exp_numbers_500k = sorted(loop2_exp_numbers_500k)
    print(f"{loop2_exp_numbers_500k=}")

    # ---- Get unique experiment numbers from full data ----
    loop2_exp_numbers_full = adata_loop2_full.obs.experiment_number.unique().tolist()
    loop2_exp_numbers_full = sorted(loop2_exp_numbers_full)
    print(f"{loop2_exp_numbers_full=}")

### Concatenate with data from loop 1.

In [19]:
if RUN_LOOP2:
    # ---- Concatenate 500k subset ----
    adata_loop2_concat_500k = ad.concat((adata_loop1_concat_500k, adata_loop2_500k), merge="same", uns_merge="same")
    adata_loop2_concat_500k.obs_names_make_unique()
    print(f"{adata_loop2_concat_500k=}")

    # ---- Concatenate full data ----
    adata_loop2_concat_full = ad.concat((adata_loop1_concat_full, adata_loop2_full), merge="same", uns_merge="same")
    adata_loop2_concat_full.obs_names_make_unique()
    print(f"{adata_loop2_concat_full=}")


### Get unique experiment numbers for concatenated data.

In [20]:
if RUN_LOOP2:
    # ---- Get unique experiment numbers from concatenated subset ----
    loop2_exp_numbers_concat_500k = adata_loop2_concat_500k.obs.experiment_number.astype(int).unique().tolist()
    loop2_exp_numbers_concat_500k = sorted(loop2_exp_numbers_concat_500k)
    print(f"Loop 2 500k subset concat {len(loop2_exp_numbers_concat_500k)} -> {loop2_exp_numbers_concat_500k=}")

    # ---- Get unique experiment numbers from concatenated full data ----
    loop2_exp_numbers_concat_full = adata_loop2_concat_full.obs.experiment_number.astype(int).unique().tolist()
    loop2_exp_numbers_concat_full = sorted(loop2_exp_numbers_concat_full)
    print(f"Loop 2 full concat {len(loop2_exp_numbers_concat_full)} -> {loop2_exp_numbers_concat_full=}")

### Handle column type for saving.

In [30]:
# ---- Define function for sanitizing adata ----
def sanitize_anndata(adata: ad.AnnData, convert_to: str = "string") -> ad.AnnData:
    for df, name in [(adata.obs, "obs"), (adata.var, "var")]:
        for col in df.columns:
            if df[col].dtype == "object":
                if convert_to == "string":
                    # Convert to pandas nullable string dtype (safe for HDF5 writing)
                    df[col] = df[col].astype("string")
                elif convert_to == "category":
                    df[col] = df[col].astype("category")
                else:
                    raise ValueError("convert_to must be 'string' or 'category'")
    return adata

if RUN_LOOP2:
    # ---- Sanitizing concatenated data ----
    adata_loop2_concat_full = sanitize_anndata(adata_loop2_concat_full)
    adata_loop2_concat_500k = sanitize_anndata(adata_loop2_concat_500k)

### Write data from loop2.

In [ ]:
# ---- Define output paths ----
loop2_dump_dir = "../../project_folder/output/loops/data/loop2/replicate"
loop2_500k_res_out_path = os.path.join(loop2_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop2_full_res_out_path = os.path.join(loop2_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop2_500k_out_path = os.path.join(loop2_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop2_full_out_path = os.path.join(loop2_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Write 500k subset ----
if RUN_LOOP2:
    print(f"Writing Loop 2 data to disk...")
    adata_loop2_500k.write_h5ad(loop2_500k_res_out_path)
    adata_loop2_full.write_h5ad(loop2_full_res_out_path)
    adata_loop2_concat_500k.write_h5ad(loop2_500k_out_path)
    adata_loop2_concat_full.write_h5ad(loop2_full_out_path)
    print(f"All data written!")

# ---- Read 500k subset ----
else:
    print(f"Reading Loop 2 data from disk...")
    adata_loop2_500k = ad.read_h5ad(loop2_500k_res_out_path)
    adata_loop2_full = ad.read_h5ad(loop2_full_res_out_path)
    adata_loop2_concat_500k = ad.read_h5ad(loop2_500k_out_path)
    adata_loop2_concat_full = ad.read_h5ad(loop2_full_out_path)
    print(f"All data read!")

## Loop 2.5.

### Read Data.

In [22]:
if RUN_LOOP2p5:
    # ---- Load 500k subset ----
    adata_loop2p5_500k_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop2p5_logicle_500k.h5ad"
    adata_loop2p5_500k = ad.read_h5ad(adata_loop2p5_500k_path)
    print(f"{adata_loop2p5_500k=}")

    # ---- Load 500k subset ----
    adata_loop2p5_full_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop2p5_logicle.h5ad"
    adata_loop2p5_full = ad.read_h5ad(adata_loop2p5_full_path)
    print(f"{adata_loop2p5_full=}")


### Get Loop 2.5 experiment numbers.

In [23]:
if RUN_LOOP2p5:
    # ---- Get unique experiment numbers from 500k subset ----
    loop2p5_exp_numbers_500k = adata_loop2p5_500k.obs.experiment_number.unique().tolist()
    loop2p5_exp_numbers_500k = sorted(loop2p5_exp_numbers_500k)
    print(f"{loop2p5_exp_numbers_500k=}")

    # ---- Get unique experiment numbers from full data ----
    loop2p5_exp_numbers_full = adata_loop2p5_full.obs.experiment_number.unique().tolist()
    loop2p5_exp_numbers_full = sorted(loop2p5_exp_numbers_full)
    print(f"{loop2p5_exp_numbers_full=}")


### Concatenate data.

In [24]:
if RUN_LOOP2p5:
    # ---- Concatenate 500k subset ----
    adata_loop2p5_concat_500k = ad.concat((adata_loop2_concat_500k, adata_loop2p5_500k), merge="same", uns_merge="same")
    adata_loop2p5_concat_500k.obs_names_make_unique()
    print(f"{adata_loop2p5_concat_500k=}")

    # ---- Concatenate full data ----
    adata_loop2p5_concat_full = ad.concat((adata_loop2_concat_full, adata_loop2p5_full), merge="same", uns_merge="same")
    adata_loop2p5_concat_full.obs_names_make_unique()
    print(f"{adata_loop2p5_concat_full=}")


### Get unique experiment numbers from concatenated data.

In [25]:
if RUN_LOOP2p5:
    # ---- Get unique experiment numbers from concatenated subset ----
    loop2p5_exp_numbers_concat_500k = adata_loop2p5_concat_500k.obs.experiment_number.astype(int).unique().tolist()
    loop2p5_exp_numbers_concat_500k = sorted(loop2p5_exp_numbers_concat_500k)
    print(f"Loop 2.5 500k subset concat {len(loop2p5_exp_numbers_concat_500k)} -> {loop2p5_exp_numbers_concat_500k=}")

    # ---- Get unique experiment numbers from concatenated full data ----
    loop2p5_exp_numbers_concat_full = adata_loop2p5_concat_full.obs.experiment_number.astype(int).unique().tolist()
    loop2p5_exp_numbers_concat_full = sorted(loop2p5_exp_numbers_concat_full)
    print(f"Loop 2.5 full concat {len(loop2p5_exp_numbers_concat_full)} -> {loop2p5_exp_numbers_concat_full=}")

### Sanitize data.

In [26]:
if RUN_LOOP2p5:
    # ---- Sanitizing concatenated data ----
    adata_loop2p5_concat_full = sanitize_anndata(adata_loop2p5_concat_full)
    adata_loop2p5_concat_500k = sanitize_anndata(adata_loop2p5_concat_500k)

### Write data to disk.

In [16]:
# ---- Define output paths ----
loop2p5_dump_dir = "../../project_folder/output/loops/data/loop2p5/replicate"
loop2p5_500k_res_out_path = os.path.join(loop2p5_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop2p5_full_res_out_path = os.path.join(loop2p5_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop2p5_500k_out_path = os.path.join(loop2p5_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop2p5_full_out_path = os.path.join(loop2p5_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Write 500k subset ----
if RUN_LOOP2p5:
    print(f"Writing Loop 2.5 data to disk...")
    adata_loop2p5_500k.write_h5ad(loop2p5_500k_res_out_path)
    adata_loop2p5_full.write_h5ad(loop2p5_full_res_out_path)
    adata_loop2p5_concat_500k.write_h5ad(loop2p5_500k_out_path)
    adata_loop2p5_concat_full.write_h5ad(loop2p5_full_out_path)
    print(f"All data written!")

# ---- Read 500k subset ----
else:
    print(f"Reading Loop 2.5 data from disk...")
    adata_loop2p5_500k = ad.read_h5ad(loop2p5_500k_res_out_path)
    adata_loop2p5_full = ad.read_h5ad(loop2p5_full_res_out_path)
    adata_loop2p5_concat_500k = ad.read_h5ad(loop2p5_500k_out_path)
    adata_loop2p5_concat_full = ad.read_h5ad(loop2p5_full_out_path)
    print(f"All data read!")

Reading Loop 2.5 data from disk...
All data read!


## Loop 3.

### Load input data.

In [27]:
if RUN_LOOP3:
    # ---- Load 500k subset ----
    adata_loop3_500k_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop3_logicle_500k.h5ad"
    adata_loop3_500k = ad.read_h5ad(adata_loop3_500k_path)
    print(f"{adata_loop3_500k=}")

    # ---- Load 500k subset ----
    adata_loop3_full_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop3_logicle.h5ad"
    adata_loop3_full = ad.read_h5ad(adata_loop3_full_path)
    print(f"{adata_loop3_full=}")


### Get experiment numbers from loop 3.

In [28]:
if RUN_LOOP3:
    # ---- Get unique experiment numbers from 500k subset ----
    loop3_exp_numbers_500k = adata_loop3_500k.obs.experiment_number.unique().tolist()
    loop3_exp_numbers_500k = sorted(loop3_exp_numbers_500k)
    print(f"{loop3_exp_numbers_500k=}")

    # ---- Get unique experiment numbers from full data ----
    loop3_exp_numbers_full = adata_loop3_full.obs.experiment_number.unique().tolist()
    loop3_exp_numbers_full = sorted(loop3_exp_numbers_full)
    print(f"{loop3_exp_numbers_full=}")


### Prepare Additional Molecules for Previous Loops.

In [32]:
if RUN_LOOP3:
    additional_molecules = ['il7_[ng_ml]', 'mcsf_[ng_ml]', 'ly_cocktail_[ul/well]']
    adata_loop2p5_concat_500k.obs[additional_molecules] = 0.0
    adata_loop2p5_concat_full.obs[additional_molecules] = 0.0

### Concatenate with Loop 2.5 Data.

In [33]:
if RUN_LOOP3:
    # ---- Concatenate 500k subset ----
    adata_loop3_concat_500k = ad.concat((adata_loop2p5_concat_500k, adata_loop3_500k), merge="same", uns_merge="same")
    adata_loop3_concat_500k.obs_names_make_unique()
    print(f"{adata_loop3_concat_500k=}")

    # ---- Concatenate full data ----
    adata_loop3_concat_full = ad.concat((adata_loop2p5_concat_full, adata_loop3_full), merge="same", uns_merge="same")
    adata_loop3_concat_full.obs_names_make_unique()
    print(f"{adata_loop3_concat_full=}")


### Get experiment numbers for concatenated data.

In [34]:
if RUN_LOOP3:
    # ---- Get unique experiment numbers from concatenated subset ----
    loop3_exp_numbers_concat_500k = adata_loop3_concat_500k.obs.experiment_number.astype(int).unique().tolist()
    loop3_exp_numbers_concat_500k = sorted(loop3_exp_numbers_concat_500k)
    print(f"Loop 3 500k subset concat {len(loop3_exp_numbers_concat_500k)} -> {loop3_exp_numbers_concat_500k=}")

    # ---- Get unique experiment numbers from concatenated full data ----
    loop3_exp_numbers_concat_full = adata_loop3_concat_full.obs.experiment_number.astype(int).unique().tolist()
    loop3_exp_numbers_concat_full = sorted(loop3_exp_numbers_concat_full)
    print(f"Loop 3 full concat {len(loop3_exp_numbers_concat_full)} -> {loop3_exp_numbers_concat_full=}")

### Write to Disk.

In [35]:
# ---- Define output paths ----
loop3_dump_dir = "../../project_folder/output/loops/data/loop3/replicate"
loop3_500k_res_out_path = os.path.join(loop3_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop3_full_res_out_path = os.path.join(loop3_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop3_500k_out_path = os.path.join(loop3_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop3_full_out_path = os.path.join(loop3_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Write 500k subset ----
if RUN_LOOP3:
    # ---- Sanitize data ----
    adata_loop3_500k = sanitize_anndata(adata_loop3_500k)
    adata_loop3_full = sanitize_anndata(adata_loop3_full)
    adata_loop3_concat_500k = sanitize_anndata(adata_loop3_concat_500k)
    adata_loop3_concat_full = sanitize_anndata(adata_loop3_concat_full)

    # ---- Extract data from dictionaries ----
    adata_loop3_500k.write_h5ad(loop3_500k_res_out_path)
    adata_loop3_full.write_h5ad(loop3_full_res_out_path)
    adata_loop3_concat_500k.write_h5ad(loop3_500k_out_path)
    adata_loop3_concat_full.write_h5ad(loop3_full_out_path)
# ---- Read 500k subset ----
else:
    adata_loop3_500k = ad.read_h5ad(loop3_500k_res_out_path)
    adata_loop3_full = ad.read_h5ad(loop3_full_res_out_path)
    adata_loop3_concat_500k = ad.read_h5ad(loop3_500k_out_path)
    adata_loop3_concat_full = ad.read_h5ad(loop3_full_out_path)

## Loop 4. 

### Load input data.

In [44]:
if RUN_LOOP4:
    # ---- Load 500k subset ----
    adata_loop4_500k_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop4_logicle_500k.h5ad"
    adata_loop4_500k = ad.read_h5ad(adata_loop4_500k_path)
    print(f"{adata_loop4_500k=}")

    # ---- Load 500k subset ----
    adata_loop4_full_path = "../../project_folder/data/gdrive/LPHO014/Loop2_Loop2p5_Loop3_Loop4_Loop4p5/Loop4_logicle.h5ad"
    adata_loop4_full = ad.read_h5ad(adata_loop4_full_path)
    print(f"{adata_loop4_full=}")
    

adata_loop4_500k=AnnData object with n_obs × n_vars = 500000 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'il7_[ng_ml]', 'mcsf_[ng_ml]', 'ly_cocktail_[ul/well]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps'
    layers: 'posit

### Get experiment numbers from loop 4.

In [45]:
if RUN_LOOP4:
    # ---- Get unique experiment numbers from 500k subset ----
    loop4_exp_numbers_500k = adata_loop4_500k.obs.experiment_number.unique().tolist()
    loop4_exp_numbers_500k = sorted(loop4_exp_numbers_500k)
    print(f"{loop4_exp_numbers_500k=}")

    # ---- Get unique experiment numbers from full data ----
    loop4_exp_numbers_full = adata_loop4_full.obs.experiment_number.unique().tolist()
    loop4_exp_numbers_full = sorted(loop4_exp_numbers_full)
    print(f"{loop4_exp_numbers_full=}")

loop4_exp_numbers_500k=['259']
loop4_exp_numbers_full=['259']


### Concatenate with Loop 3 Data.

In [51]:
if RUN_LOOP4:
    # ---- Concatenate 500k subset ----
    adata_loop4_concat_500k = ad.concat((adata_loop3_concat_500k, adata_loop4_500k), merge="same", uns_merge="same")
    adata_loop4_concat_500k.obs_names_make_unique()
    print(f"{adata_loop4_concat_500k=}")

    # ---- Concatenate full data ----
    adata_loop4_concat_full = ad.concat((adata_loop3_concat_full, adata_loop4_full), merge="same", uns_merge="same")
    adata_loop4_concat_full.obs_names_make_unique()
    print(f"{adata_loop4_concat_full=}")

/home/icb/alessandro.palma/miniconda3/envs/sc_exp_design_12/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/tmp/ipykernel_2502332/2236141185.py:4: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['50217-0-1', '7846-1-1', '27424-2-1', '166607-4-1', '90475-0-1']
  adata_loop4_concat_500k.obs_names_make_unique()


adata_loop4_concat_500k=AnnData object with n_obs × n_vars = 2499947 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD', 'il7_[ng_ml]', 'mcsf_[ng_ml]', 'ly_cocktail_[ul/well]'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps'
    layers

/home/icb/alessandro.palma/miniconda3/envs/sc_exp_design_12/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/tmp/ipykernel_2502332/2236141185.py:9: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['0-0-1', '0-0-2', '0-0-3', '5-0-1', '5-0-2']
  adata_loop4_concat_full.obs_names_make_unique()


adata_loop4_concat_full=AnnData object with n_obs × n_vars = 70110606 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD', 'il7_[ng_ml]', 'mcsf_[ng_ml]', 'ly_cocktail_[ul/well]'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps'
    layer

### Get experiment numbers for concatenated data.

In [57]:
if RUN_LOOP4:
    # ---- Get unique experiment numbers from concatenated subset ----
    loop4_exp_numbers_concat_500k = adata_loop4_concat_500k.obs.experiment_number.astype(int).unique().tolist()
    loop4_exp_numbers_concat_500k = sorted(loop4_exp_numbers_concat_500k)
    print(f"Loop 4 500k subset concat {len(loop4_exp_numbers_concat_500k)} -> {loop4_exp_numbers_concat_500k=}")

    # ---- Get unique experiment numbers from concatenated full data ----
    loop4_exp_numbers_concat_full = adata_loop4_concat_full.obs.experiment_number.astype(int).unique().tolist()
    loop4_exp_numbers_concat_full = sorted(loop4_exp_numbers_concat_full)
    print(f"Loop 4 full concat {len(loop4_exp_numbers_concat_full)} -> {loop4_exp_numbers_concat_full}")

Loop 4 500k subset concat 143 -> loop4_exp_numbers_concat_500k=[40, 41, 42, 43, 53, 54, 55, 65, 66, 67, 77, 78, 79, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 133, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 157, 158, 159, 160, 161, 162, 163, 164, 188, 189, 205, 206, 207, 208, 209, 210, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1111, 1112, 1113, 1114, 1115, 1116, 1117, 1118, 1119, 1120, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1169, 1170, 1171, 1172, 1173, 1174, 1175, 1176, 1177, 1178, 1179, 1180, 1181, 1182, 1188, 1189, 2138, 2139, 4138, 4139]
Loop 4 full concat 143 -> [40, 41, 42, 43, 53, 54, 55, 65, 66, 67, 77, 78, 79, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125,

### Write to Disk.

In [16]:
# ---- Define output paths ----
loop4_dump_dir = "../../project_folder/output/loops/data/loop4/replicate"
loop4_500k_res_out_path = os.path.join(loop4_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop4_full_res_out_path = os.path.join(loop4_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop4_500k_out_path = os.path.join(loop4_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop4_full_out_path = os.path.join(loop4_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Write 500k subset ----
if RUN_LOOP4:
    # ---- Sanitize data ----
    adata_loop4_500k = sanitize_anndata(adata_loop4_500k)
    adata_loop4_full = sanitize_anndata(adata_loop4_full)
    adata_loop4_concat_500k = sanitize_anndata(adata_loop4_concat_500k)
    adata_loop4_concat_full = sanitize_anndata(adata_loop4_concat_full)

    # ---- Extract data from dictionaries ----
    adata_loop4_500k.write_h5ad(loop4_500k_res_out_path)
    adata_loop4_full.write_h5ad(loop4_full_res_out_path)
    adata_loop4_concat_500k.write_h5ad(loop4_500k_out_path)
    adata_loop4_concat_full.write_h5ad(loop4_full_out_path)
# ---- Read 500k subset ----
else:
    adata_loop4_500k = ad.read_h5ad(loop4_500k_res_out_path)
    adata_loop4_full = ad.read_h5ad(loop4_full_res_out_path)
    adata_loop4_concat_500k = ad.read_h5ad(loop4_500k_out_path)
    adata_loop4_concat_full = ad.read_h5ad(loop4_full_out_path)

## Loop 5. 

### Load input data.

In [6]:
if RUN_LOOP5:
    # ---- Load 500k subset ----
    adata_loop_500k_path = "../../project_folder/data/gdrive/LPHO015/LPHO015/Results_Fig6/results_Leiden_500k.h5ad"
    adata_loop5_500k = ad.read_h5ad(adata_loop_500k_path)
    print(f"{adata_loop5_500k=}")

    # ---- Load 500k subset ----
    adata_loop5_full_path = "../../project_folder/data/gdrive/LPHO015/LPHO015/Results_Fig6/results_logicle.h5ad"
    adata_loop5_full = ad.read_h5ad(adata_loop5_full_path)
    print(f"{adata_loop5_full=}")
    

adata_loop5_500k=AnnData object with n_obs × n_vars = 464207 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'il7_[ng_ml]', 'mcsf_[ng_ml]', 'ly_cocktail_[ul/well]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD', 'leiden'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'leiden', 'leiden_colors', 'meta', 

In [10]:
adata_loop5_500k.obs["experiment_number"].unique()

['267', '265', '272', '276', '270', ..., '273', '278', '280', '262', '275']
Length: 20
Categories (20, object): ['261', '262', '263', '264', ..., '277', '278', '279', '280']

In [11]:
adata_loop5_full.obs["experiment_number"].unique()

['262', '263', '264', '265', '266', ..., '269', '268', '272', '278', '274']
Length: 20
Categories (20, object): ['261', '262', '263', '264', ..., '277', '278', '279', '280']

### Get experiment numbers from loop 4.

In [13]:
if RUN_LOOP5:
    # ---- Get unique experiment numbers from 500k subset ----
    loop5_exp_numbers_500k = adata_loop5_500k.obs.experiment_number.unique().tolist()
    loop5_exp_numbers_500k = sorted(loop5_exp_numbers_500k)
    print(f"{loop5_exp_numbers_500k=}")

    # ---- Get unique experiment numbers from full data ----
    loop5_exp_numbers_full = adata_loop5_full.obs.experiment_number.unique().tolist()
    loop5_exp_numbers_full = sorted(loop5_exp_numbers_full)
    print(f"{loop5_exp_numbers_full=}")

loop5_exp_numbers_500k=['261', '262', '263', '264', '265', '266', '267', '268', '269', '270', '271', '272', '273', '274', '275', '276', '277', '278', '279', '280']
loop5_exp_numbers_full=['261', '262', '263', '264', '265', '266', '267', '268', '269', '270', '271', '272', '273', '274', '275', '276', '277', '278', '279', '280']


### Concatenate with Loop 4 Data.

In [18]:
if RUN_LOOP5:
    # ---- Concatenate 500k subset ----
    adata_loop5_concat_500k = ad.concat((adata_loop4_concat_500k, adata_loop5_500k), merge="same", uns_merge="same")
    adata_loop5_concat_500k.obs_names_make_unique()
    print(f"{adata_loop4_concat_500k=}")

    # ---- Concatenate full data ----
    adata_loop5_concat_full = ad.concat((adata_loop4_concat_full, adata_loop5_full), merge="same", uns_merge="same")
    adata_loop5_concat_full.obs_names_make_unique()
    print(f"{adata_loop5_concat_full=}")

/home/icb/alessandro.palma/miniconda3/envs/sc_exp_design_12/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/tmp/ipykernel_1994485/2102437648.py:4: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['67499-26-1', '20065-27-1', '88768-26-1', '37753-25-1', '154340-29-1']
  adata_loop5_concat_500k.obs_names_make_unique()


adata_loop4_concat_500k=AnnData object with n_obs × n_vars = 2499947 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD', 'il7_[ng_ml]', 'mcsf_[ng_ml]', 'ly_cocktail_[ul/well]'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps'
    layers

/home/icb/alessandro.palma/miniconda3/envs/sc_exp_design_12/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/tmp/ipykernel_1994485/2102437648.py:9: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['366-0-1', '366-0-2', '495-0-1', '495-0-2', '514-0-1']
  adata_loop5_concat_full.obs_names_make_unique()


adata_loop5_concat_full=AnnData object with n_obs × n_vars = 78268470 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD', 'il7_[ng_ml]', 'mcsf_[ng_ml]', 'ly_cocktail_[ul/well]'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps'
    layer

### Get experiment numbers for concatenated data.

In [21]:
if RUN_LOOP5:
    # ---- Get unique experiment numbers from concatenated subset ----
    loop5_exp_numbers_concat_500k = adata_loop5_concat_500k.obs.experiment_number.astype(int).unique().tolist()
    loop5_exp_numbers_concat_500k = sorted(loop5_exp_numbers_concat_500k)
    print(f"Loop 5 500k subset concat {len(loop5_exp_numbers_concat_500k)} -> {loop5_exp_numbers_concat_500k=}")

    # ---- Get unique experiment numbers from concatenated full data ----
    loop5_exp_numbers_concat_full = adata_loop5_concat_full.obs.experiment_number.astype(int).unique().tolist()
    loop5_exp_numbers_concat_full = sorted(loop5_exp_numbers_concat_full)
    print(f"Loop 5 full concat {len(loop5_exp_numbers_concat_full)} -> {loop5_exp_numbers_concat_full}")

Loop 5 500k subset concat 163 -> loop5_exp_numbers_concat_500k=[40, 41, 42, 43, 53, 54, 55, 65, 66, 67, 77, 78, 79, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 133, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 157, 158, 159, 160, 161, 162, 163, 164, 188, 189, 205, 206, 207, 208, 209, 210, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1111, 1112, 1113, 1114, 1115, 1116, 1117, 1118, 1119, 1120, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1169, 1170, 1171, 1172, 1173, 1174, 1175, 1176, 1177, 1178, 1179, 1180, 1181, 1182, 1188, 1189, 2138, 2139, 4138, 4139]
Loop 5 full concat 163 -> [40, 41, 42, 43, 53, 54, 55, 65, 66, 67, 77, 78, 79,

In [27]:
print(adata_loop4_concat_full.shape)
print(adata_loop5_concat_full.shape)

(70110606, 20)
(78268470, 20)


### Write to Disk.

In [31]:
# ---- Define output paths ----
loop5_dump_dir = "../../project_folder/output/loops/data/loop5"
loop5_500k_res_out_path = os.path.join(loop5_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop5_full_res_out_path = os.path.join(loop5_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop5_500k_out_path = os.path.join(loop5_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop5_full_out_path = os.path.join(loop5_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Write 500k subset ----
if RUN_LOOP5:
    # ---- Sanitize data ----
    adata_loop5_500k = sanitize_anndata(adata_loop5_500k)
    adata_loop5_full = sanitize_anndata(adata_loop5_full)
    adata_loop5_concat_500k = sanitize_anndata(adata_loop5_concat_500k)
    adata_loop5_concat_full = sanitize_anndata(adata_loop5_concat_full)

    # ---- Extract data from dictionaries ----
    adata_loop5_500k.write_h5ad(loop5_500k_res_out_path)
    adata_loop5_full.write_h5ad(loop5_full_res_out_path)
    adata_loop5_concat_500k.write_h5ad(loop5_500k_out_path)
    adata_loop5_concat_full.write_h5ad(loop5_full_out_path)
    
# ---- Read 500k subset ----
else:
    adata_loop5_500k = ad.read_h5ad(loop5_500k_res_out_path)
    adata_loop5_full = ad.read_h5ad(loop5_full_res_out_path)
    adata_loop5_concat_500k = ad.read_h5ad(loop5_500k_out_path)
    adata_loop5_concat_full = ad.read_h5ad(loop5_full_out_path)

/home/icb/alessandro.palma/miniconda3/envs/sc_exp_design_12/lib/python3.12/site-packages/anndata/_io/utils.py:272: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)
/home/icb/alessandro.palma/miniconda3/envs/sc_exp_design_12/lib/python3.12/site-packages/anndata/_io/utils.py:272: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)
/home/icb/alessandro.palma/miniconda3/envs/sc_exp_design_12/lib/python3.12/site-packages/anndata/_io/utils.py:272: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)
